# SVM-RBF & Hierarchical Clustering — Sara
Complete reproducible analysis on the Breast Cancer Wisconsin Diagnostic dataset.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, json, os
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.inspection import permutation_importance
from sklearn.datasets import load_breast_cancer

os.makedirs('results/SVM', exist_ok=True)
os.makedirs('results/Hierarchical_Clustering', exist_ok=True)

def load_project_data():
    for path in [Path("breast_cancer.csv"), Path("/content/breast_cancer.csv"), Path("/mnt/data/breast_cancer.csv")]:
        if path.exists():
            data = pd.read_csv(path)
            if {"id", "diagnosis"}.issubset(data.columns):
                print(f"Loaded: {path}")
                return data
    raw = load_breast_cancer(as_frame=True)
    features = raw.data.copy()
    def project_name(name):
        if name.startswith("mean "): return name[5:].replace(" ", "_") + "_mean"
        if name.endswith(" error"): return name[:-6].replace(" ", "_") + "_se"
        if name.startswith("worst "): return name[6:].replace(" ", "_") + "_worst"
        return name.replace(" ", "_")
    features.columns = [project_name(c) for c in features.columns]
    data = features.copy()
    data.insert(0, "id", np.arange(1, len(data)+1))
    data.insert(1, "diagnosis", np.where(raw.target.to_numpy()==0, "M", "B"))
    print("breast_cancer.csv not found; loaded the equivalent scikit-learn dataset.")
    return data

df = load_project_data()
X = df.drop(columns=['id','diagnosis'])
y = (df.diagnosis == 'M').astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)

pipe=Pipeline([('scaler',StandardScaler()),('svc',SVC(kernel='rbf',probability=True,random_state=42))])
params={'svc__C':[0.1,1,10,100], 'svc__gamma':[0.001,0.01,0.1,1]}
cv=StratifiedKFold(5,shuffle=True,random_state=42); gs=GridSearchCV(pipe,params,scoring='roc_auc',cv=cv,n_jobs=-1); gs.fit(Xtr,ytr)
model=gs.best_estimator_; pred=model.predict(Xte); prob=model.predict_proba(Xte)[:,1]; cm=confusion_matrix(yte,pred); tn,fp,fn,tp=cm.ravel()
metrics={'Accuracy':accuracy_score(yte,pred),'Precision':precision_score(yte,pred),'Sensitivity':recall_score(yte,pred),'Specificity':tn/(tn+fp),'F1':f1_score(yte,pred),'ROC-AUC':roc_auc_score(yte,prob),'CV ROC-AUC':gs.best_score_}
pd.DataFrame(metrics.items(),columns=['Metric','Value']).to_csv('results/SVM/svm_metrics.csv',index=False)
pd.DataFrame(cm,index=['Actual Benign','Actual Malignant'],columns=['Predicted Benign','Predicted Malignant']).to_csv('results/SVM/svm_confusion_matrix.csv')
pd.DataFrame([{'C':gs.best_params_['svc__C'],'gamma':gs.best_params_['svc__gamma']}]).to_csv('results/SVM/svm_best_parameters.csv',index=False)


In [ ]:
# CM plot
fig,ax=plt.subplots(figsize=(5,4)); im=ax.imshow(cm); ax.set_xticks([0,1],['Benign','Malignant']); ax.set_yticks([0,1],['Benign','Malignant']); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('SVM-RBF Confusion Matrix');
for i in range(2):
 for j in range(2): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=14)
fig.tight_layout(); fig.savefig('results/SVM/svm_confusion_matrix.png',dpi=180); plt.close(fig)
# ROC
fpr,tpr,_=roc_curve(yte,prob); fig,ax=plt.subplots(figsize=(5,4)); ax.plot(fpr,tpr,label=f"AUC = {metrics['ROC-AUC']:.3f}"); ax.plot([0,1],[0,1],'--'); ax.set(xlabel='False Positive Rate',ylabel='True Positive Rate',title='SVM-RBF ROC Curve'); ax.legend(); fig.tight_layout(); fig.savefig('results/SVM/svm_roc_curve.png',dpi=180); plt.close(fig)
# Permutation importance
pi=permutation_importance(model,Xte,yte,n_repeats=30,random_state=42,scoring='roc_auc'); imp=pd.DataFrame({'Feature':X.columns,'Importance':pi.importances_mean,'Std':pi.importances_std}).sort_values('Importance',ascending=False); imp.head(10).to_csv('results/SVM/svm_permutation_top10.csv',index=False)
top=imp.head(10).sort_values('Importance'); fig,ax=plt.subplots(figsize=(7,5)); ax.barh(top.Feature,top.Importance,xerr=top.Std); ax.set(xlabel='Decrease in test ROC-AUC after permutation',title='SVM-RBF Permutation Importance — Top 10'); fig.tight_layout(); fig.savefig('results/SVM/svm_permutation_top10.png',dpi=180); plt.close(fig)
# HC all standardized
Xs=StandardScaler().fit_transform(X); hc=AgglomerativeClustering(n_clusters=2,linkage='ward'); lab=hc.fit_predict(Xs)
ari=adjusted_rand_score(y,lab); nmi=normalized_mutual_info_score(y,lab); sil=silhouette_score(Xs,lab)
ct=pd.crosstab(df.diagnosis,lab); pd.DataFrame([['ARI',ari],['NMI',nmi],['Silhouette',sil]],columns=['Metric','Value']).to_csv('results/Hierarchical_Clustering/hierarchical_metrics.csv',index=False); ct.to_csv('results/Hierarchical_Clustering/hierarchical_cluster_composition.csv')
# dendrogram
Z=linkage(Xs,method='ward'); fig,ax=plt.subplots(figsize=(10,5)); dendrogram(Z,truncate_mode='lastp',p=30,leaf_rotation=90,leaf_font_size=8,ax=ax); ax.set(title='Hierarchical Clustering — Ward Dendrogram',xlabel='Clustered sample groups',ylabel='Ward distance'); fig.tight_layout(); fig.savefig('results/Hierarchical_Clustering/hierarchical_dendrogram.png',dpi=180); plt.close(fig)
# PCA cluster and diagnosis
pca=PCA(2); pcs=pca.fit_transform(Xs); pd.DataFrame([['PC1',pca.explained_variance_ratio_[0]],['PC2',pca.explained_variance_ratio_[1]],['PC1+PC2',pca.explained_variance_ratio_.sum()]],columns=['Component','ExplainedVariance']).to_csv('results/Hierarchical_Clustering/hierarchical_pca_variance.csv',index=False)
fig,ax=plt.subplots(figsize=(6,5));
for k in [0,1]:
 m=lab==k; ax.scatter(pcs[m,0],pcs[m,1],s=22,alpha=.7,label=f'Cluster {k}')
ax.set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',ylabel=f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)',title='Hierarchical Clusters in PCA Space'); ax.legend(); fig.tight_layout(); fig.savefig('results/Hierarchical_Clustering/hierarchical_pca_clusters.png',dpi=180); plt.close(fig)
fig,ax=plt.subplots(figsize=(6,5));
for val,name in [(0,'Benign'),(1,'Malignant')]:
 m=y.values==val; ax.scatter(pcs[m,0],pcs[m,1],s=22,alpha=.7,label=name)
ax.set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',ylabel=f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)',title='Diagnosis in PCA Space'); ax.legend(); fig.tight_layout(); fig.savefig('results/Hierarchical_Clustering/hierarchical_pca_diagnosis.png',dpi=180); plt.close(fig)
print('best',gs.best_params_,gs.best_score_); print(metrics); print(cm); print('HC',ari,nmi,sil); print(ct); print(imp.head(10).to_string(index=False))
